In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path().resolve().parent
CACHE_DIR = PROJECT_ROOT / "data" / "raw"

CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["KAGGLEHUB_CACHE"] = str(CACHE_DIR)

import kagglehub

path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
print(path)

In [ ]:
from pathlib import Path
import pandas as pd
import os

path = Path(path)

df_fake = pd.read_csv(path / "Fake.csv")
df_true = pd.read_csv(path / "True.csv")

In [ ]:
df_fake.head()

In [ ]:
df_fake = df_fake.assign(label="0")
df_true = df_true.assign(label="1")

In [ ]:
df_fake = df_fake.drop(columns=["subject", "date", "title"])
df_true = df_true.drop(columns=["subject", "date", "title"])

In [ ]:
df_news = pd.concat([df_fake, df_true], ignore_index=True)
df_news

In [ ]:
df_news.isnull().sum()

In [ ]:
df_news.duplicated().sum()

In [ ]:
df_news.drop_duplicates(inplace=True)

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

nltk.download('wordnet', "/kaggle/working/nltk_data/")
nltk.download('omw-1.4', "/kaggle/working/nltk_data/")
nltk.download('punkt_tab', "/kaggle/working/nltk_data/")
nltk.download('stopwords', "/kaggle/working/nltk_data/")

In [ ]:
import zipfile
import os

zip_paths = [
    "/kaggle/working/nltk_data/corpora/wordnet.zip",
    "/kaggle/working/nltk_data/corpora/omw-1.4.zip",
    "/kaggle/working/nltk_data/tokenizers/punkt_tab.zip",
    "/kaggle/working/nltk_data/corpora/stopwords.zip"
]

extract_dir = "/kaggle/working/nltk_data/corpora"

os.makedirs(extract_dir, exist_ok=True)

for z in zip_paths:
    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
nltk.data.path.append("/kaggle/working/nltk_data/")

In [ ]:
def process_text(text):
    text = re.sub(
        r"\s+", " ", text, flags=re.I
    )  # Remove extra white space from text

    text = re.sub(
        r"\W", " ", str(text)
    )  # Remove all the special characters from text

    text = re.sub(
        r"\s+[a-zA-Z]\s+", " ", text
    )  # Remove all single characters from text

    text = re.sub(
        r"[^a-zA-Z\s]", "", text
    )  # Remove any character that isn't alphabetical

    text = text.lower()

    words = word_tokenize(text)

    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]

    stop_words = set(stopwords.words("english"))
    Words = [word for word in words if word not in stop_words]

    Words = [word for word in Words if len(word) > 3]

    indices = np.unique(Words, return_index=True)[1]
    cleaned_text = np.array(Words)[np.sort(indices)].tolist()

    return cleaned_text

In [ ]:
df_news.to_csv("news.csv")

In [ ]:
x = df_news.drop("label", axis=1)
y = df_news.label

In [ ]:
texts = list(x["text"])

In [ ]:
cleaned_text = [process_text(text) for text in texts]

In [ ]:
print(cleaned_text[:5])

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    cleaned_text, y, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

tokenizer = Tokenizer()
tokenizer.fit_on_texts(x_train)
word_idx = tokenizer.word_index
v = len(word_idx)
print("the size of vocab =", v)
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = 200
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

In [ ]:
y.value_counts()

In [ ]:
from keras.models import Sequential
from keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Input,
    GlobalMaxPooling1D,
    Dropout,
)
from tensorflow.keras.models import Model
from keras import optimizers
import numpy as np
from tensorflow.keras.optimizers import Adam

In [ ]:
input = Input(shape=(maxlen,))
learning_rate = 0.0001
x = Embedding(v + 1, 100)(input)
x = Dropout(0.5)(x)
x = LSTM(150, return_sequences=True)(x)
x = Dropout(0.5)(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(2, activation="softmax")(x)

model = Model(input, x)

optimizer = Adam(learning_rate=learning_rate)

model.compile(
    optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

In [ ]:
import tensorflow as tf

y_train_one_hot = tf.keras.utils.to_categorical(y_train_encoded)
y_test_one_hot = tf.keras.utils.to_categorical(y_test_encoded)

In [26]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath="best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
)

history = model.fit(
    x_train,
    y_train_one_hot,
    epochs=15,
    batch_size=128,
    validation_data=(x_test, y_test_one_hot),
    callbacks=[checkpoint],
)

242/242 ━━━━━━━━━━━━━━━━━━━━ 62s 250ms/step - accuracy: 0.6243 - loss: 0.6355 - val_accuracy: 0.9151 - val_loss: 0.4335
Epoch 2/15
242/242 ━━━━━━━━━━━━━━━━━━━━ 62s 256ms/step - accuracy: 0.9551 - loss: 0.1733 - val_accuracy: 0.9766 - val_loss: 0.1431
Epoch 3/15
242/242 ━━━━━━━━━━━━━━━━━━━━ 64s 263ms/step - accuracy: 0.9838 - loss: 0.0594 - val_accuracy: 0.9833 - val_loss: 0.0760
Epoch 4/15
242/242 ━━━━━━━━━━━━━━━━━━━━ 63s 260ms/step - accuracy: 0.9905 - loss: 0.0363 - val_accuracy: 0.9829 - val_loss: 0.0567
Epoch 5/15
242/242 ━━━━━━━━━━━━━━━━━━━━ 62s 256ms/step - accuracy: 0.9938 - loss: 0.0247 - val_accuracy: 0.9847 - val_loss: 0.0487
Epoch 6/15
242/242 ━━━━━━━━━━━━━━━━━━━━ 61s 253ms/step - accuracy: 0.9951 - loss: 0.0200 - val_accuracy: 0.9871 - val_loss: 0.0698
Epoch 7/15
242/242 ━━━━━━━━━━━━━━━━━━━━ 63s 262ms/step - accuracy: 0.9972 - loss: 0.0125 - val_accuracy: 0.9871 - val_loss: 0.0448
Epoch 8/15
242/242 ━━━━━━━━━━━━━━━━━━━━ 63s 262ms/step - accuracy: 0.9982 - loss: 0.0091 - val

In [27]:
# import matplotlib.pyplot as plt

# plt.plot(history.history["accuracy"])
# plt.plot(history.history["val_accuracy"])
# plt.title("Model accuracy")
# plt.xlabel("Epoch")
# plt.ylabel("Accuracy")
# plt.legend(["Train", "Test"], loc="upper left")
# plt.show()

# plt.plot(history.history["loss"])
# plt.plot(history.history["val_loss"])
# plt.title("Model loss")
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.legend(["Train", "Test"], loc="upper left")
# plt.show()

In [28]:
# loss, accuracy = model.evaluate(x_test, y_test_one_hot)

# print("Test Loss:", loss)
# print("Test Accuracy:", accuracy)

In [29]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import confusion_matrix
# import numpy as np


# y_pred_probs = model.predict(x_test)
# y_pred_labels = np.argmax(y_pred_probs, axis=1)
# y_true_labels = np.argmax(y_test_one_hot, axis=1)
# conf_matrix = confusion_matrix(y_true_labels, y_pred_labels)
# plt.figure(figsize=(8, 6))
# sns.heatmap(
#     conf_matrix,
#     annot=True,
#     fmt="d",
#     cmap="Blues",
#     xticklabels=["Fake", "Real"],
#     yticklabels=["Fake", "Real"],
# )
# plt.xlabel("Predicted")
# plt.ylabel("True")
# plt.title("Confusion Matrix")
# plt.show()

In [30]:
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint

df = pd.read_csv("../data/raw/datasets/mahdimashayekhi/fake-news-detection-dataset/versions/1/fake_or_real_news.csv")
label_map = {"fake": 0, "real": 1}
y_new = df["label"].str.lower().map(label_map).astype(int).values
texts = df["text"].astype(str).tolist()

processed = [" ".join(process_text(t)) for t in texts]

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

seqs = tokenizer.texts_to_sequences(processed)
X_new = pad_sequences(seqs, maxlen=maxlen)

y_new_onehot = tf.keras.utils.to_categorical(y_new, num_classes=2)

model = load_model("best_model.keras")

checkpoint = ModelCheckpoint(
    filepath="finetuned_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
)

model.fit(
    X_new,
    y_new_onehot,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[checkpoint],
)

Epoch 1/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 16s 187ms/step - accuracy: 0.7052 - loss: 0.6519 - val_accuracy: 0.7822 - val_loss: 0.5735
Epoch 2/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 14s 180ms/step - accuracy: 0.8155 - loss: 0.4537 - val_accuracy: 0.8264 - val_loss: 0.4987
Epoch 3/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 15s 182ms/step - accuracy: 0.8650 - loss: 0.3325 - val_accuracy: 0.8366 - val_loss: 0.4101
Epoch 4/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 15s 185ms/step - accuracy: 0.9079 - loss: 0.2348 - val_accuracy: 0.8627 - val_loss: 0.3627
Epoch 5/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 15s 182ms/step - accuracy: 0.9501 - loss: 0.1502 - val_accuracy: 0.8721 - val_loss: 0.3308
Epoch 6/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 15s 189ms/step - accuracy: 0.9647 - loss: 0.1029 - val_accuracy: 0.8777 - val_loss: 0.3242
Epoch 7/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 13s 164ms/step - accuracy: 0.9769 - loss: 0.0690 - val_accuracy: 0.8721 - val_loss: 0.3211
Epoch 8/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 13s 162ms/step - accuracy: 0.9844 - loss: 0.0498 - val_accu

In [32]:
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint

df = pd.read_csv(
    "../data/raw/datasets/saurabhshahane/fake-news-classification/versions/77/WELFake_Dataset.csv"
).dropna()
df = df.drop_duplicates()
y_new = df["label"].astype(int).map({1: 0, 0: 1}).values
texts = df["text"].astype(str).tolist()

processed = [" ".join(process_text(t)) for t in texts]

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

seqs = tokenizer.texts_to_sequences(processed)
X_new = pad_sequences(seqs, maxlen=maxlen)

y_new_onehot = tf.keras.utils.to_categorical(y_new, num_classes=2)

model = load_model("finetuned_model.keras")

checkpoint = ModelCheckpoint(
    filepath="v3_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1,
)

history = model.fit(
    X_new,
    y_new_onehot,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[checkpoint],
    verbose=1,
)

Epoch 1/5
895/895 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.8983 - loss: 0.2540
Epoch 1: val_loss improved from None to 0.20049, saving model to v3_model.keras
895/895 ━━━━━━━━━━━━━━━━━━━━ 174s 193ms/step - accuracy: 0.9144 - loss: 0.2152 - val_accuracy: 0.9428 - val_loss: 0.2005
Epoch 2/5
895/895 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.9516 - loss: 0.1336
Epoch 2: val_loss improved from 0.20049 to 0.16761, saving model to v3_model.keras
895/895 ━━━━━━━━━━━━━━━━━━━━ 149s 167ms/step - accuracy: 0.9523 - loss: 0.1316 - val_accuracy: 0.9504 - val_loss: 0.1676
Epoch 3/5
895/895 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.9655 - loss: 0.0967
Epoch 3: val_loss improved from 0.16761 to 0.14568, saving model to v3_model.keras
895/895 ━━━━━━━━━━━━━━━━━━━━ 162s 181ms/step - accuracy: 0.9659 - loss: 0.0970 - val_accuracy: 0.9534 - val_loss: 0.1457
Epoch 4/5
895/895 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.9756 - loss: 0.0731
Epoch 4: val_loss improved from 0.14568 to 

In [ ]:
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv("fake_or_real_news.csv")

X_raw = df["text"].astype(str).tolist()

label_map = {"fake": "0", "real": "1"}
y_int = df["label"].str.lower().map(label_map).astype(int).values

processed_docs = [" ".join(process_text(t)) for t in X_raw]

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

seqs = tokenizer.texts_to_sequences(processed_docs)
X = pad_sequences(seqs, maxlen=maxlen)

y_one_hot = tf.keras.utils.to_categorical(y_int, num_classes=2)

model = load_model("best_model.keras")

loss, acc = model.evaluate(X, y_one_hot, verbose=0)
print("CSV accuracy:", acc)

In [ ]:
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML
import pickle
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv("../data/raw/datasets/mahdimashayekhi/fake-news-detection-dataset/versions/1/fake_or_real_news.csv")
texts = df["text"].astype(str).tolist()
labels = df["label"].astype(str).tolist()

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

model = load_model("finetuned_model.keras")


def predict_fn(text_list):
    processed = [" ".join(process_text(t)) for t in text_list]
    seqs = tokenizer.texts_to_sequences(processed)
    X = pad_sequences(seqs, maxlen=maxlen)
    return model.predict(X)


explainer = LimeTextExplainer(class_names=["fake", "real"])
i = 5814
sample = texts[i]
label = labels[i]
exp = explainer.explain_instance(sample, predict_fn, num_features=20)

print(label)
display(HTML(exp.as_html()))